# Part 2 -- Why the failures fail: the cause list

Part 1 built a small pipeline on NeoQA (whole articles, one embedder, cosine
top-10), measured it, and counted the real failures. The
tempting next move is to grab a fix off the shelf -- a bigger model, a
re-ranker, hybrid search. This part does the opposite: read every failure
and name the causes first, because which fix is worth building depends on
why the failures happen.

Code cells are excerpts from the real scripts; outputs are pasted from the
actual runs. Question examples are paraphrased, never quoted.

## 1. How do you find causes without guessing?

Step by step:

- **Shuffle the failures.** Read them in that order, so the first few don't
  shape how you read the rest.
- **Write a free note on each one:** what seems to have pushed the needed
  article out of the top-10.
- **Group the notes into suspected causes.**
- **Turn each suspected cause into a small script rule** with a fixed
  threshold -- frozen before any counting.
- **Let the script label every question -- successes included.** A cause
  you only look for in failures will always be "confirmed"; the successes
  are the control group.

In [ ]:
def leans_to_one_half(q):                  # one cause, written as a rule
    gap = cos(q, easiest_needed(q)) - cos(q, binding(q))
    return gap >= 0.05                     # threshold frozen before counting

labels = {q.id: [rule(q) for rule in RULES] for q in questions}

reading pass   171 failures, shuffled (seed 20260703), free notes on each
               last NEW cause: failure 45 of 171 -- everything after repeats
rules          every candidate frozen as a threshold, applied to all 266
               95 successes labeled by the same rules -- the control group

## 2. What are the causes?

Six causes survived. In plain words:

- **the question leans to one half** -- it needs two articles; its single
  vector sits close to one, and the other sinks.
- **a wrong event captures the words** -- the corpus has near-identical
  events; the query's words fit another event better than the needed one.
- **the fact drowns in its article** -- the needed fact is one sentence; the
  whole-article vector is about something else.
- **nothing to grab** -- the question never names its second article, so
  keywords miss it too.
- **a broken question** -- refers to context no retriever can see; a dataset
  artifact, kept apart so it never inflates a real cause.
- **near-copies crowd the list** -- reworded copies fill the top-10 and
  starve the needed article.

In [ ]:
show_cause_counts(labels)

cause                               failures   successes
the question leans to one half        39%          4%
a wrong event captures the words      53%         28%
the fact drowns in its article        27%       (see below)
nothing to grab                       20%          4%
a broken question                      4%          3%
near-copies crowd the list            58%         23%

a typical failure carries about two causes -- the columns do not sum

One row needs a note: the drown rule cannot fire on a success by
construction (it needs a deeply-ranked article to lift), so its check uses
the continuous value instead -- scoring by best sentence clearly helps the
failures' needed articles and does not help the successes'. And a row that
fires often on successes too, like the wrong-event one, is partly a
background condition of this crowded corpus: real, but secondary.

Here is one failure that has a single cause:

In [ ]:
q = failures[j]        # the clearest one-cause case; paraphrased output
show_failure(q)

question (paraphrased): what did the person who advised young athletes on
core strength and avoiding injuries at a training session reveal about a
gymnastics champion's decision to retire?

needs two events:            the retirement (event 0) + the training session (event 1)
listed as complete evidence: exactly 1 pair of copies -- ev0 copy 2 + ev1 copy 2

where that pair ranked:
  sports_13_1-ev0-2    cos 0.776    rank  7     inside the top-10
  sports_13_1-ev1-2    cos 0.642    rank 88     the failure

the training session has 12 reworded copies; the 11 that are NOT listed
as evidence for this question ALL rank above the one that is
keyword ranking (BM25) puts sports_13_1-ev1-2 at rank 1

This is the near-copy problem in a single case. Part 1's warning -- copies
look alike but are not interchangeable -- is the whole failure here: the
embedder ranks the training-session copies as a block and cannot tell which
one carries the needed detail. Keyword ranking, which matches exact words,
does not have this problem.

## 3. Which causes survive the checks?

When you read a failure, it is easy to guess a cause. To know if a guess
is real, we put each one through two checks.

**Check 1 -- does the cause tell failures and successes apart?**

A cause of failure should be common in failures and rare in successes.
That is what the successes column in the table above is for: run each
cause's rule on the successes too, not just the failures. A real cause
shows a wide gap between the two columns. A rule that fires about as often
on successes is a background property of the corpus, not a cause of failure.

Reading the failures, this candidate looked convincing -- "the evidence
lives in only a couple of listed copy-pairs, so there are fewer chances to
land a pair in the top-10." Run it over the successes too:

In [ ]:
check_over_successes("evidence lives in <= 2 listed pairs")

candidate cause: evidence lives in <= 2 listed pairs ("little redundancy")
  failures     6 of 171    (4%)
  successes    7 of 95     (7%)

We named this candidate "little redundancy": the question's evidence lives
in only a couple of listed copy-pairs. But it shows up more often among the
successes than the failures. A cause of failure cannot be more common in
successes -- so we dropped it.

**Check 2 -- if you remove the cause, do the failures come back?**

Check 1 shows a cause and failure appear together. Appearing together does
not prove the cause is what breaks the question. Check 2 tests that
directly:

- **Undo the mechanism** -- change the scoring so the cause cannot happen.
- **Re-rank every question** under the new scoring.
- **Count what moved** -- failures fixed, and successes broken.

Undoing looks different for each cause:

- **The fact drowns** because each article is scored by one vector for its
  whole text, averaging a single relevant sentence away. Undo it by scoring
  each article on its best-matching sentence -- now that sentence can carry
  the article.
- **Near-copies crowd** because reworded copies of the needed events fill
  the ten slots. Undo it by dropping the copies not listed as valid
  evidence, then re-rank.

In [ ]:
rescore(undo="the fact drowns")     # score each article by its best sentence
rescore(undo="near-copies crowd")   # drop each event's unlisted copies

the fact drowns   -- score each article by its best sentence
   171 -> 130 failures     60 fixed, 19 successes broken
near-copies crowd -- drop the unlisted copies of the events
   171 -> 151 failures     20 fixed,  0 successes broken

The drowning fact is a real cause: a large block of failures turns into
successes -- though some successes break, so it is not a free fix. The
near-copies gave a surprising result. They sit on more failures than any
other cause, yet removing them fixes only a fraction. A cause can appear in
a failure without being what broke it.

**Co-occurrence is not cause.** Check 1 dropped a cause that looked
convincing. Check 2 showed our most common cause fixes only a few of the
failures it appears in.

### Aside: retriever's fault, or the dataset's?

Before fixes, one caveat -- and no new measurement, just a second reading of
the causes above. Each one has a general mechanism any RAG system can hit,
and an amplifier that is NeoQA's own construction.

- **leans to one half** -- general: any two-part question squeezed into one
  vector leans to one part. NeoQA amplifies it: its questions fuse two
  events into one sentence, one vivid, one vague.
- **wrong event captures the words** -- general in any single-topic corpus
  (one company's news, one product's docs). NeoQA's event density is
  extreme.
- **the fact drowns** -- general to any index that stores one vector per
  long document.
- **nothing to grab** -- general to questions that describe a thing instead
  of naming it.
- **a broken question** -- purely the dataset's.
- **near-copies crowd the list** -- the mechanism hits any corpus without
  dedup; the twelve-copies-per-event uniformity is NeoQA's.

The cause types are what transfer to your system. The percentages are one
dataset under one embedder -- don't carry them anywhere.

## 4. So which fix is worth building?

Each surviving cause points at one: the keyword signal points at hybrid
search, the drowning fact at sentence-level scoring, the near-copies at
dedup. Pricing them -- how many failures each fix would actually return,
overlaps included -- is Part 3's job. First, the leftovers this list does not cover:

In [ ]:
leftovers(labels)

no cause fires:                       9 of 171
wrong-event is the only cause:       19 more     (the weak separator)
no fix on this list targets these:   28

(To be continued: pricing the fixes, and building the first one.)